Colorisation

1.   Imports

In [ ]:
# Imports et montage du drive
import os
import re
import cv2
import gc
import glob
import shutil
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from google.colab import drive
import shutil
from io import BytesIO

# Montage du Google Drive
drive.mount('/content/drive')

# Chemins globaux
PROJECT_NAME = "coco_pix2pix"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints_pix2pix_color"
DATASET_LOCAL_DIR = "/content/dataset"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
#Accès au Git et installation des dépendances
%cd /content/
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

# Installation des paquets requis par le dépôt officiel
!pip install -q dominate visdom wandb scipy tqdm opencv-python

In [ ]:
def est_en_couleur(content_images, seuil=8):
    try:
        img = Image.open(BytesIO(content_images)).convert('RGB')
        r, g, b = img.split()

        r_arr, g_arr, b_arr = np.array(r), np.array(g), np.array(b)

        rg_diff = np.mean(np.abs(r_arr - g_arr))
        rb_diff = np.mean(np.abs(r_arr - b_arr))

        # Si l'écart moyen est trop faible, l'image est considérée comme grise
        if rg_diff < seuil and rb_diff < seuil:
            return False
        return True
    except Exception:
        return False

# On nettoie les anciens dossiers pour éviter les mélanges
if os.path.exists(DATASET_LOCAL_DIR):
    shutil.rmtree(DATASET_LOCAL_DIR)

os.makedirs(os.path.join(DATASET_LOCAL_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(DATASET_LOCAL_DIR, "val"), exist_ok=True)

# Compteur global d'essais pour garantir un seed unique à chaque requête
id_unique = 0

# Téléchargement des données d'entraînement : 1000 images
img_train_telechargees = 0
pbar_train = tqdm(total=1000, desc="Téléchargement Train")

while img_train_telechargees < 1000:
    try:
        # Utilisation d'un identifiant unique incrémental pour le seed
        url = f"https://picsum.photos/seed/dataset_color_{id_unique}/256/256"
        id_unique += 1 # On l'incrémente immédiatement pour la prochaine requête

        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            # On ne garde l'image que si elle possède de vraies couleurs et on rejette les N&B
            if est_en_couleur(response.content):
                nom_fichier = f"{DATASET_LOCAL_DIR}/train/image_{img_train_telechargees}.jpg"
                with open(nom_fichier, "wb") as f:
                    f.write(response.content)
                img_train_telechargees += 1
                pbar_train.update(1)
    except Exception:
        continue
pbar_train.close()

# Téléchargement des données de validation : 100 images
img_val_telechargees = 0
pbar_val = tqdm(total=100, desc="Téléchargement Val")

while img_val_telechargees < 100:
    try:
        # On continue d'incrémenter le même id_unique.
        url = f"https://picsum.photos/seed/dataset_color_{id_unique}/256/256"
        id_unique += 1

        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            if est_en_couleur(response.content):
                nom_fichier = f"{DATASET_LOCAL_DIR}/val/image_val_{img_val_telechargees}.jpg"
                with open(nom_fichier, "wb") as f:
                    f.write(response.content)
                img_val_telechargees += 1
                pbar_val.update(1)
    except Exception:
        continue
pbar_val.close()

# Vérification finale
nb_train = len(os.listdir(os.path.join(DATASET_LOCAL_DIR, "train")))
nb_val = len(os.listdir(os.path.join(DATASET_LOCAL_DIR, "val")))

In [ ]:
#Logique de reprise
checkpoint_project_dir = os.path.join(CHECKPOINT_DIR, PROJECT_NAME)
last_epoch = 0
continue_flag = ""

if os.path.exists(checkpoint_project_dir):
    files = os.listdir(checkpoint_project_dir)
    epochs = [int(re.findall(r'(\d+)_net_G.pth', f)[0]) for f in files if 'net_G.pth' in f and 'latest' not in f]
    if epochs:
        last_epoch = max(epochs)
        continue_flag = "--continue_train"
        print(f"Dernier checkpoint : Epoch {last_epoch}")

resume_epoch = last_epoch + 1

In [ ]:
#Lancement de l'entraînement
if nb_train > 0:
    print(f"Lancement de l'entraînement : Epoch {resume_epoch}...")
    # On se déplace impérativement dans le repo avant de lancer train.py
    %cd /content/pytorch-CycleGAN-and-pix2pix

    !python train.py --dataroot {DATASET_LOCAL_DIR} \
                     --name {PROJECT_NAME} \
                     --model pix2pix \
                     --direction AtoB \
                     --dataset_mode colorization \
                     --input_nc 1 \
                     --output_nc 2 \
                     --n_epochs 200 \
                     --n_epochs_decay 200 \
                     --batch_size 8 \
                     --checkpoints_dir {CHECKPOINT_DIR} \
                     --save_epoch_freq 10 \
                     --lambda_L1 30 \
                     {continue_flag} \
                     --gan_mode vanilla
else:
    print("Le dossier train est vide.")

In [ ]:
#Script de validation manuel
import os
import glob
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import color

%cd /content/pytorch-CycleGAN-and-pix2pix
from models.networks import define_G

PATH_TO_WEIGHTS = "/content/drive/MyDrive/checkpoints_pix2pix_color/coco_pix2pix/250_net_G.pth"
VAL_IMAGES_DIR = "/content/dataset/val/"

if not os.path.exists(PATH_TO_WEIGHTS):
    print(f"Impossible de trouver les poids du modèle.")
else:
    print("Chargement du générateur Pix2Pix (Unet 256)...")
    netG = define_G(input_nc=1, output_nc=2, ngf=64, netG='unet_256', norm='batch', use_dropout=False, init_type='normal', init_gain=0.02)

    state_dict = torch.load(PATH_TO_WEIGHTS, map_location='cuda:0')
    if hasattr(state_dict, '_metadata'): del state_dict._metadata
    netG.load_state_dict(state_dict)
    netG.eval()
    netG.cuda()

    # 1. On récupère TOUTES les images JPG du dossier de validation
    images_dispos = sorted(glob.glob(os.path.join(VAL_IMAGES_DIR, "*.jpg")))

    if len(images_dispos) == 0:
        print("Aucune image originale trouvée dans /content/dataset/val/")
    else:
        # 2. On choisit le nombre d'images à afficher (ici les 5 premières, ou change l'index)
        nb_images_a_afficher = min(100, len(images_dispos))
        print(f"Génération de la galerie pour {nb_images_a_afficher} images de validation...\n")

        for i, img_path in enumerate(images_dispos[:nb_images_a_afficher]):
            img_bgr = cv2.imread(img_path)
            if img_bgr is None: continue

            # Traitement de l'image
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, (256, 256))

            # Conversion Espace Lab via skimage
            img_lab_orig = color.rgb2lab(img_resized / 255.0)
            img_L = img_lab_orig[:, :, 0] # Luminance N&B

            # Normalisation et Tenseur pour PyTorch
            tens_L = torch.from_numpy(img_L).float().unsqueeze(0).unsqueeze(0).cuda()
            tens_L_norm = (tens_L / 50.0) - 1.0

            # Inférence du modèle
            with torch.no_grad():
                output_ab_norm = netG(tens_L_norm)

            # Dénormalisation des couleurs prédites
            output_ab = (output_ab_norm.squeeze(0).cpu().float().numpy().transpose(1, 2, 0) + 1.0) * 110.0 - 110.0

            # Recomposition Lab
            lab_fake = np.zeros((256, 256, 3))
            lab_fake[:, :, 0] = img_L
            lab_fake[:, :, 1] = output_ab[:, :, 0]
            lab_fake[:, :, 2] = output_ab[:, :, 1]

            # Conversion finale en Vrai RGB
            rgb_fake = (color.lab2rgb(lab_fake) * 255).astype(np.uint8)

            # Affichage de la ligne de comparaison
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            axes[0].imshow(img_L, cmap='gray')
            axes[0].set_title(f"Image {i+1} : Entrée")
            axes[0].axis('off')

            axes[1].imshow(rgb_fake)
            axes[1].set_title("Colorisation")
            axes[1].axis('off')

            axes[2].imshow(img_resized)
            axes[2].set_title("Vérité terrain")
            axes[2].axis('off')

            plt.show()